In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
sc.pp.normalize_total(adata_scraw)
sc.pp.log1p(adata_scraw)
adata_scrna.raw = adata_scraw.copy()

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")
adata_scrna

In [ ]:
adata_scrna.obs['cellannot'] = adata_scrna.obs['majority_voting'].replace({
    'T cells': 'Lymphoid',
    'Resident NK': 'Lymphoid',
    'Circulating NK/NKT': 'Lymphoid',
    'Macrophages': 'Myeloid',
    'Mono+mono derived cells': 'Myeloid',
    'Endothelial cells': 'Endothelial',
    'Fibroblasts': 'Stellete',
    'Hepatocytes': 'Hepatocyte',
    'Cholangiocytes' : 'Cholangiocyte'
})

sc.pl.umap(adata_scrna, color=['cellannot'])

In [ ]:
adata_scraw.obs['cellannot']=adata_scrna.obs['cellannot']

In [ ]:
adata_scstell = adata_scraw[adata_scraw.obs['cellannot'].isin(['Stellete'])].copy()
adata_scstell

In [ ]:
adata_scstell.write("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_stel01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_scstell, target_sum=1e4)
sc.pp.log1p(adata_scstell)
adata_scstell.raw = adata_scstell
sc.pp.highly_variable_genes(adata_scstell, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_scstell, max_value=10)
# Run PCA
sc.tl.pca(adata_scstell, svd_solver='arpack')
print(adata_scstell.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_scstell, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_scstell.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_scstell, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scstell, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scstell)
sc.pl.umap(adata_scstell, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scstell.obsm['X_pca'],adata_scstell.obs, 'sample')

# Harmony 결과를 저장
adata_scstell.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scstell, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scstell)

In [ ]:
sc.pl.umap(adata_scstell,color='state')

In [ ]:
adata_scstell.obs['state'].value_counts()